# Upserting YouTube transcripts

In this following exercises, you'll create a chatbot that can answer questions
about YouTube videos by ingesting video transcripts and additional metadata
into your 'pinecone-datacamp' index.

To start, you'll prepare data from the youtube_rag_data.csv file
and upsert the vectors with all of their metadata into the 'pinecone-datacamp' index.
The data is provided in the DataFrame youtube_df.

```python
Here's an example transcript from the youtube_df DataFrame:
id:
35Pdoyi6ZoQ-t0.0

title:
Training and Testing an Italian BERT - Transformers From Scratch #4

text:
Hi, welcome to the video. So this is the fourth video in a Transformers from Scratch
mini series. So if you haven't been following along, we've essentially covered what
you can see on the screen. So we got some data. We built a tokenizer with it...

url:
https://youtu.be/35Pdoyi6ZoQ

published:
01-01-2024
```

```python
==============================================
print(df.info())
<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 693 to 327
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          200 non-null    object
 1   blob        200 non-null    object
 2   channel_id  200 non-null    object
 3   end         200 non-null    int64
 4   published   200 non-null    object
 5   start       200 non-null    int64
 6   text        200 non-null    object
 7   title       200 non-null    object
 8   url         200 non-null    object
dtypes: int64(2), object(7)
memory usage: 15.6+ KB
None

==============================================
print(df.head())
                       id  ...                           url
693    SGazDb8o-to-t236.0  ...  https://youtu.be/SGazDb8o-to
56     35Pdoyi6ZoQ-t922.6  ...  https://youtu.be/35Pdoyi6ZoQ
299   x1lAcT3xl5M-t1142.0  ...  https://youtu.be/x1lAcT3xl5M
1027   ODdKC30dT8c-t834.9  ...  https://youtu.be/ODdKC30dT8c
208    x1lAcT3xl5M-t212.0  ...  https://youtu.be/x1lAcT3xl5M

[5 rows x 9 columns]
```

In [1]:
import pandas as pd
from file_helper import get_file_path
from openai_helper import create_embeddings, create_embeddings_with_model
from pinecone_helper import get_index

csv_path = get_file_path("youtube_rag_data_small.csv")
index_name = "pinecone-datacamp"
batch_limit = 100
namespace = "youtube_rag_dataset"
emb_model = "text-embedding-3-small"

# n=200: 随机选择 200 行
# random_state=42: 固定随机结果，方便测试
df = pd.read_csv(csv_path).sample(
    n=200, random_state=42
)  # 每次运行都会得到相同的 200 行

index = get_index(index_name)

In [2]:
for start in range(0, len(df), batch_limit):
    batch = df.iloc[start : start + batch_limit]

    # Extract the metadata from each row
    metadatas = [
        {
            "text_id": row["id"],
            "text": row["text"],
            "title": row["title"],
            "url": row["url"],
            "published": row["published"],
        }
        for _, row in batch.iterrows()
    ]
    texts = batch["text"].tolist()

    # Use the original dataset ID as the Pinecone vector ID
    ids = batch["id"].astype(str).tolist()

    # Encode texts using OpenAI
    embeds = create_embeddings(texts)

    # Create Pinecone vectors
    vectors = [
        (id_, embedding, metadata)
        for id_, embedding, metadata in zip(ids, embeds, metadatas)
    ]

    # Upsert vectors to the correct namesapce
    index.upsert(vectors=vectors, namespace="youtube_rag_dataset")

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=400, metric='cosine', namespaces=2)


# Building a retrieval function

A key process in the Retrieval Augmented Generation (RAG) workflow
is retrieving data from the database.
In this exercise, you'll design a custom function called retrieve()
that will perform this crucial process in the final exercise of the course.

In [3]:
# Define a retrieve function that takes four arguments:
# query, top_k, namespace, emb_model
def retrieve(query, top_k, namespace, emb_model):
    # Encode the input query using OpenAI
    query_emb = create_embeddings_with_model(texts=query, model=emb_model)

    # Query the index using the query_emb
    docs = index.query(
        vector=query_emb, top_k=top_k, namespace=namespace, include_metadata=True
    )

    retrieved_docs = []
    sources = []
    for doc in docs["matches"]:
        retrieved_docs.append(doc["metadata"]["text"])
        sources.append((doc["metadata"]["title"], doc["metadata"]["url"]))

    return retrieved_docs, sources

In [4]:
# Test
documents, sources = retrieve(
    query="How to build next-level Q&A with OpenAI",
    top_k=3,
    namespace=namespace,
    emb_model=emb_model,
)

print(documents)
print(sources)

["and our model into a pipeline, into a Q&A pipeline. So again, we get this pipeline from the Transformers library. So we come down here, do from Transformers, import pipeline. And now what we want to do is just initialize a pipeline object. So to do that, we just write pipeline. And then in here, what we need to add is a model type. So obviously, you can see up here, we have all of these different tasks. So summarization, text generation and so on. The Transformers library needs to understand, or this pipeline object needs to understand which one of those pipelines or functions we are intending to use. So to tell it that we want to do question answering, we just write question answering. And that basically sets the wrapper of the pipeline to handle question answering formats. So we'll see our input and for our input, we will be passing a context and a question. So we'll see that it will convert into the right structure that we need for question answering, which is the CLS context sepa

# RAG questions answering function

You're almost there! The final piece in the RAG workflow is
to integrate the retrieved documents with a question-answering model.

A prompt_with_context_builder() function has already been defined
and made available to you.
This function takes the documents retrieved from the Pinecone index,
and integrates them into a prompt that the question-answering model can ingest:

In [ ]:
from openai_helper import client
from pinecone_helper import get_index


def prompt_with_context_builder(query, docs) -> str:
    delim = "\n\n---\n\n"
    prompt_start = "Answer the question based on the context below.\n\nContext:\n"
    prompt_end = f"\n\nQuestion: {query}\nAnswer:"

    prompt = prompt_start + delim.join(docs) + prompt_end
    return prompt


index = get_index(index_name)
query = "How to build next-level Q&A with OpenAI?"

# Retrieve the top three most similar documents and their resources
documents, sources = retrieve(
    query=query,
    top_k=3,
    namespace=namespace,
    emb_model=emb_model,
)

prompt_with_context = prompt_with_context_builder(query, documents)
print(prompt_with_context)

Answer the question based on the context below.

Context:
and our model into a pipeline, into a Q&A pipeline. So again, we get this pipeline from the Transformers library. So we come down here, do from Transformers, import pipeline. And now what we want to do is just initialize a pipeline object. So to do that, we just write pipeline. And then in here, what we need to add is a model type. So obviously, you can see up here, we have all of these different tasks. So summarization, text generation and so on. The Transformers library needs to understand, or this pipeline object needs to understand which one of those pipelines or functions we are intending to use. So to tell it that we want to do question answering, we just write question answering. And that basically sets the wrapper of the pipeline to handle question answering formats. So we'll see our input and for our input, we will be passing a context and a question. So we'll see that it will convert into the right structure that we ne

In [8]:
def question_answering(prompt, sources, chat_model):
    sys_prompt = "You are a helpful assistant that always answers questions."

    response = client.responses.create(
        model=chat_model,
        instructions=sys_prompt,
        input=prompt,
        temperature=0,
    )

    answer = response.output_text.strip()
    answer += "\n\nSources:"
    for source in sources:
        answer += "\n" + source[0] + ": " + source[1]

    return answer


answer = question_answering(
    prompt=prompt_with_context,
    sources=sources,
    chat_model="gpt-4o-mini",
)

print(answer)

To build next-level Q&A with OpenAI, you can follow these steps:

1. **Utilize the Transformers Library**: Start by importing the necessary components from the Transformers library, specifically the `pipeline` for question answering.

2. **Initialize the Pipeline**: Create a pipeline object by specifying the model type as "question answering". This sets up the framework to handle Q&A tasks.

3. **Prepare Input Data**: Structure your input by providing a context and a question. The pipeline will format this input correctly for processing.

4. **Leverage Pre-trained Models**: Explore the Hugging Face Models page to find pre-trained models specifically for question answering. You can download these models to quickly start experimenting.

5. **Fine-tuning (Optional)**: If you have specific use cases, consider fine-tuning the model on your dataset to improve performance.

6. **Experiment with Datasets**: Use datasets like SQUAD to understand how to extract answers, including determining the